In [56]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
import time

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MUSIXMATCH_API_KEY = os.getenv("MUSIXMATCH_API_KEY")

BASE_URL = "https://api.musixmatch.com/ws/1.1"

client = OpenAI(api_key=OPENAI_API_KEY)


In [57]:
def search_tracks(title, artist=None):
    url = f"{BASE_URL}/track.search"

    params = {
        "apikey": MUSIXMATCH_API_KEY,
        "q_track": title,
        "page_size": 10,
        "s_track_rating": "desc"
    }

    if artist:
        params["q_artist"] = artist

    response = requests.get(url, params=params)
    data = response.json()

    body = data.get("message", {}).get("body", {})
    track_list = body.get("track_list", [])

    return track_list


In [58]:
def search_by_lyrics_snippet(snippet):
    url = f"{BASE_URL}/track.search"
    
    params = {
        "apikey": MUSIXMATCH_API_KEY,
        "q_lyrics": snippet,
        "page_size": 10,
        "s_track_rating": "desc"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Safe extraction
    body = data.get("message", {}).get("body", {})
    
    # Sometimes body is a list
    if isinstance(body, list):
        track_list = body
    else:
        track_list = body.get("track_list", [])
    
    if not track_list:
        return None
    
    # Handle both formats
    first_item = track_list[0]
    track = first_item.get("track", first_item)
    
    return {
        "track_id": track.get("track_id"),
        "track_name": track.get("track_name"),
        "artist_name": track.get("artist_name"),
        "album_name": track.get("album_name"),
        "isrc": track.get("track_isrc"),
        "rating": track.get("track_rating")
    }


In [59]:
def get_lyrics(track_id):
    url = f"{BASE_URL}/track.lyrics.get"

    params = {
        "apikey": MUSIXMATCH_API_KEY,
        "track_id": track_id
    }

    response = requests.get(url, params=params)
    data = response.json()

    message = data.get("message", {})
    body = message.get("body", {})

    # Handle inconsistent Musixmatch responses
    if isinstance(body, list):
        # No lyrics returned
        return None

    lyrics = body.get("lyrics")

    if lyrics and lyrics.get("lyrics_body"):
        return lyrics.get("lyrics_body")

    return None


In [ ]:
import json

def ai_extract_song_info(user_input):
    prompt = f"""
    A user entered this text to search for a song:

    "{user_input}"

    The text might be:
    - a full title
    - part of a title
    - artist name
    - a line from the lyrics

    If the text looks like lyrics, determine the most likely song title and artist.

    For worship songs, return the most popular recorded version.

    Respond ONLY in valid JSON:
    {{
        "title": "...",
        "artist": null or "..."
    }}
    """

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )

    content = response.choices[0].message.content.strip()
    content = content.replace("```json", "").replace("```", "")

    return json.loads(content)


In [61]:
def find_song_ai(user_input):
    print("\n🔎 Understanding request with AI...")
        
    print("AI time:", time.time() - start)
    
    # Step 1 — AI extraction
    song_info = ai_extract_song_info(user_input)
    title = song_info.get("title")
    artist = song_info.get("artist")

    print(f"AI Guess → Title: {title} | Artist: {artist}")

    # Step 2 — First search (title + artist)
    tracks = search_tracks(title, artist)

    # Step 3 — Fallback: search by title only
    if not tracks:
        print("🔁 No results with artist, retrying with title only...")
        tracks = search_tracks(title)

    if not tracks:
        print("❌ No tracks found.")
        return None

    # Step 4 — Find most popular track WITH lyrics
    for item in tracks:
        track = item["track"]
        track_id = track["track_id"]

        lyrics = get_lyrics(track_id)

        if lyrics:
            result = {
                "artist": track["artist_name"],
                "title": track["track_name"],
                "album": track["album_name"],
                "isrc": track.get("track_isrc"),
                "rating": track.get("track_rating"),
                "lyrics": lyrics
            }

            print("\n=== Most Popular Match With Lyrics ===")
            print("Artist :", result["artist"])
            print("Title  :", result["title"])
            print("Album  :", result["album"])
            print("ISRC   :", result["isrc"])
            print("Rating :", result["rating"])

            return result

    print("⚠️ Tracks found but no lyrics available (plan limitation or indexing issue).")
    return None


In [62]:
def display_song(result):
    if not result:
        print("No song found.")
        return
    
    print("\n" + "="*50)
    print(f"Title  : {result['title']}")
    print(f"Artist : {result['artist']}")
    print(f"Album  : {result['album']}")
    print(f"ISRC   : {result['isrc']}")
    print(f"Rating : {result['rating']}")
    print("="*50)
    print("\nLyrics:\n")
    print(result['lyrics'])
    print("="*50)


In [64]:


user_input = input("Enter song title, artist, or lyrics: ")

start = time.time()
song_info = ai_extract_song_info(user_input)
print("AI time:", time.time() - start)

start = time.time()
tracks = search_tracks(song_info["title"], song_info["artist"])
print("Search time:", time.time() - start)

start = time.time()
if tracks:
    get_lyrics(tracks[0]["track"]["track_id"])
print("Lyrics time:", time.time() - start)

result = find_song_ai(user_input)
display_song(result)


AI time: 16.796313047409058
Search time: 1.0552117824554443
Lyrics time: 0.6555314064025879

🔎 Understanding request with AI...
AI time: 0.6557323932647705
AI Guess → Title: Better Is One Day | Artist: Matt Redman

=== Most Popular Match With Lyrics ===
Artist : Matt Redman
Title  : Better Is One Day
Album  : The Friendship And The Fear
ISRC   : GBBTM9585712
Rating : 38

Title  : Better Is One Day
Artist : Matt Redman
Album  : The Friendship And The Fear
ISRC   : GBBTM9585712
Rating : 38

Lyrics:

How lovely is Your dwelling place, Oh Lord Almighty
My soul longs and even faints for You
For here my heart is satisfied, within Your presence
I sing beneath the shadow of Your wings

Better is one day in Your courts
Better is one day in Your house
Better is one day in Your courts
Than thousands elsewhere

Better is one day in Your courts
Better is one day in Your house
Better is one day in Your courts
Than thousands elsewhere

One thing I ask, and I would seek, to see Your beauty,
To find Yo